# **HuggingFace Login**

In [ ]:
from huggingface_hub import login

login()  # You will be prompted for your HF key, which will then be saved locally

# **Installs**

In [ ]:
!pip install langchain-huggingface langchain-community huggingface_hub transformers datasets langchain-yt-dlp youtube-transcript-api faiss-cpu

# **Imports**

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace, HuggingFaceEndpointEmbeddings
from langchain_community.document_loaders import YoutubeLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

# **Step 1**

# **Indexing**

In [ ]:
video_id='1a1VXDdIyrk'

loader = YoutubeLoader(
    video_id=video_id
)

document = loader.load()

In [ ]:
document

# **Text Splitter**

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=50)
chunks = text_splitter.create_documents([document[0].page_content])

In [ ]:
document = loader.load()

document

In [ ]:
chunks = text_splitter.create_documents([document[0].page_content])

In [ ]:
chunks

# **Embed Chunks | Store in VectorDB (FAISS)**

In [ ]:
embed_model = HuggingFaceEndpointEmbeddings(
    repo_id = 'ibm-granite/granite-embedding-311m-multilingual-r2',
    task = 'feature-extraction'
)

In [ ]:
vector_store = FAISS.from_documents(chunks, embed_model)

In [ ]:
vector_store.index_to_docstore_id

In [ ]:
vector_store.get_by_ids(['8642e907-fa16-4252-b5dd-88c70d9e34e6'])

# **Step 2**

# **Retriever**

In [ ]:
retriever = vector_store.as_retriever(search_type = 'similarity', search_kwargs = {'k':4})

In [ ]:
retriever

In [ ]:
retriever.invoke('What is Harness')

# **Prompt Augmentation**

In [ ]:
prompt = PromptTemplate(
    template = """
    ""
      You are a very helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say in a polite manner that you don't know about it. Don't Make up or Hallucinate Any Answer from yourself.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [ ]:
question = "What's harness engineering and how is it helpful and what's next after harness engineering?"
retrieved_docs = retriever.invoke(question)

In [ ]:
context_text = " \n ".join(rd.page_content for rd in retrieved_docs)

In [ ]:
final_prompt = prompt.invoke({'context':context_text, 'question':question})

# **Model Download**

In [44]:
!pip install -q \
langchain \
langchain-community \
langchain-huggingface \
transformers \
accelerate \
torch \
faiss-cpu \
youtube-transcript-api \
langchain-yt-dlp

In [45]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [46]:
from transformers import pipeline

pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.3,
    do_sample=True
)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [47]:
from langchain_huggingface import HuggingFacePipeline

llm = HuggingFacePipeline(
    pipeline=pipe
)

# **Generation**

In [48]:
# model = HuggingFaceEndpoint(
#     repo_id="mistralai/Mistral-7B-Instruct-v0.3",
#     task="conversational",
#     max_new_tokens=512,
#     do_sample=False,
#     repetition_penalty=1.03,
# )
# Because the inference of these models weren't available, so a model is downloaded using Torch Pipeline

chat = ChatHuggingFace(llm=llm, verbose=True)

In [ ]:
# This code is just to check the available inference servers on huggingface to run a model


# from huggingface_hub import model_info
# # Change this to the model you want to test
# info = model_info("mistralai/Mistral-7B-Instruct-v0.3", expand="inference") #
# print(info.transformers_info)


In [50]:
final_prompt

StringPromptValue(text='\n    ""\n      You are a very helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say in a polite manner that you don\'t know about it. Don\'t Make up or Hallucinate Any Answer from yourself.\n\n      doesn\'t really help us understand what it is and what it isn\'t. How exactly is harness engineering \n underneath. One clarification to be made here is that harness engineering doesn\'t necessarily \n More on them later. To put it simply, harness engineering actually existed before the term harness \n what it isn\'t. How exactly is harness engineering different from prompt engineering and context\n      Question: What\'s harness engineering and how is it helpful and what\'s next after harness engineering?\n    ')

In [49]:
output = chat.invoke(final_prompt)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [51]:
output

AIMessage(content='<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\n\n    ""\n      You are a very helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say in a polite manner that you don\'t know about it. Don\'t Make up or Hallucinate Any Answer from yourself.\n\n      doesn\'t really help us understand what it is and what it isn\'t. How exactly is harness engineering \n underneath. One clarification to be made here is that harness engineering doesn\'t necessarily \n More on them later. To put it simply, harness engineering actually existed before the term harness \n what it isn\'t. How exactly is harness engineering different from prompt engineering and context\n      Question: What\'s harness engineering and how is it helpful and what\'s next after harness engineering?\n    <|im_end|>\n<|im_start|>assistant\nBased on the information provided, harnes

In [52]:
final_output = output.content

In [53]:
final_output

'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\n\n    ""\n      You are a very helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say in a polite manner that you don\'t know about it. Don\'t Make up or Hallucinate Any Answer from yourself.\n\n      doesn\'t really help us understand what it is and what it isn\'t. How exactly is harness engineering \n underneath. One clarification to be made here is that harness engineering doesn\'t necessarily \n More on them later. To put it simply, harness engineering actually existed before the term harness \n what it isn\'t. How exactly is harness engineering different from prompt engineering and context\n      Question: What\'s harness engineering and how is it helpful and what\'s next after harness engineering?\n    <|im_end|>\n<|im_start|>assistant\nBased on the information provided, harness engineering refe

# **Building Every Component in a Chain with Lesser Code**

In [54]:
from langchain_core.runnables import RunnableParallel, RunnableSequence, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers.string import StrOutputParser

write a function to format the retriever output (list of documents)

In [55]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [56]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [57]:
parallel_chain.invoke('What is Harness Engineering?')

{'context': "doesn't really help us understand what it is and what it isn't. How exactly is harness engineering\n\nunderneath. One clarification to be made here is that harness engineering doesn't necessarily\n\nMore on them later. To put it simply, harness engineering actually existed before the term harness\n\nis that harness engineering doesn't necessarily deprecate context engineering and it certainly",
 'question': 'What is Harness Engineering?'}

In [58]:
parser = StrOutputParser()

In [59]:
chain = parallel_chain | prompt | chat | parser

In [60]:
chain.invoke('Please summarize the video')

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\n\n    ""\n      You are a very helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say in a polite manner that you don\'t know about it. Don\'t Make up or Hallucinate Any Answer from yourself.\n\n      capture the essence of something transformative that was happening in the AI industry. So the\n\nit would summarize its context to shrink it and continue working on them without having to run out\n\nevery second counts. Quick shout out to cursor. More on them later. To put it simply, harness\n\nfrom what we\'ve seen before? But first, a quick word from cursor. I\'m always trying to build on my\n      Question: Please summarize the video\n    <|im_end|>\n<|im_start|>assistant\nThe video discusses advancements in the AI industry and highlights the importance of quick progress. It mentions "cursor" as someo

**Finished, Thank You**

**Don't Forget to follow on:**

**> linkedin.com/in/hafizmusanadeem**

**> github.com/hafizmusanadeem**